# First Implementation of the RSA model 

In [44]:
import numpy as np 
import random as rd

In [53]:
V = ["dog", "cat", "rabbit"]
salience = {"dog": 0.7, "cat": 0.5, "rabbit": 0.3}
sim_dic = {"dog": [0.5,0.3], "cat": [0.5,0.3], "rabbit": [0.3,0.3]}
players = ["P1", "P2", "P3"]
roles = {"P1": "agent", "P2": "agent", "P3": "imposter"}
target = "kangaroo"
rationality = {"P1": 3, "P2": 3, "P3": 5}
C = {k: 1 for k in V}

def sim(u, t, role="agent"):
    if isinstance(t,str): 
        #print("target is string")
        if role=="agent":
            return(sim_dic[u][0])
        else:
            return(sim_dic[u][1])
    else: 
        print("FAILURE")
        return(1)


class Agent:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = target
         self.target_imp = self.get_target()
         self.P_w = {p: [2/3, 1/3] for p in players}
         self.alpha = rationality[player]

    def get_target(self):
        mv = max(salience.values())
        res = [k for k in salience if salience[k] == mv]
        return(rd.choice(res))

    def L0(self, u, p_player = [2/3, 1/3]):
        p_agent = sim(u,self.target, role="agent") * p_player[0]
        p_imposter = sim(u,self.target_imp, role="imposter") * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self):
        p_distr = []
        for u in V: 
            # in the future this might have to iterate over multiple listeners 
            # interestingly though, the agents don't care how the imposter perceives them, so this would only be a problem for the imposter
            p_u = np.exp(self.alpha * (np.log(self.L0(u)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1()
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    
    def L1(self, u, player):
        p_agent = self.S1()[u] * self.P_w[player][0]
        p_imposter = self.S1()[u] * self.P_w[player][1]
        p_sum = sum(p_agent, p_imposter)
        p_agent_norm = p_agent/p_sum
        p_imposter_norm = p_imposter/p_sum
        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_w[player] = [p_agent_norm, p_imposter_norm]
        return((p_agent_norm, p_imposter_norm))


class Imposter:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = self.get_target()
         self.P_t = salience.copy()
         self.alpha = rationality[player]
         self.target_imp = self.get_target()

    def get_target(self):
        mv = max(salience.values())
        res = [k for k in salience if salience[k] == mv]
        return(rd.choice(res))
    
    
    def L0(self, u, t, p_player = [2/3, 1/3]):
        p_agent = sim(u,t, role="agent") * p_player[0]
        p_imposter = sim(u,self.target_imp, role="imposter") * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self, t):
        p_distr = []
        for u in V: 
            # what exactly is the world? The persona the player wants to communicate? That would always be agent.
            p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1(self.target)
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    

    def L1(self, u):
        t_distr = []
        for t in V: 
            p = self.S1(t)[u] * self.P_t[t]
            t_distr.append(p)

        p_sum = sum(t_distr)
        t_dic = {t: p/p_sum for t,p in zip(V,t_distr)}

        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_t = t_dic

        return(t_dic)
    



class GameState:
    def __init__(self): 
        self.previous_words = []

    def say_word(self, word):
        self.previous_words += word

In [54]:

probs = p1.S1()
mv = max(probs.values())
res = [k for k in probs if probs[k] == mv]
rd.choice(res)

'cat'

In [57]:
p3 = Imposter("P3")

In [58]:
p3.L1("rabbit")

{'dog': np.float64(0.4666666666666667),
 'cat': np.float64(0.33333333333333337),
 'rabbit': np.float64(0.2)}